In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load the dataset
df = pd.read_csv(os.path.expanduser("~/Desktop/supply-chain-forecasting/DataCoSupplyChainDataset.csv"), 
                 encoding='latin-1')

# Quick first look
print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Shape: (180519, 53)

First 5 rows:


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [4]:
# Basic exploration
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget variable distribution:")
print(df['Late_delivery_risk'].value_counts())
print("\nDelivery Status distribution:")
print(df['Delivery Status'].value_counts())

Shape: (180519, 53)

Column names:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Produc

In [5]:
# Check cardinality of product columns
print("Category Name unique values:", df['Category Name'].nunique())
print("Department Name unique values:", df['Department Name'].nunique())
print("Product Name unique values:", df['Product Name'].nunique())
print("\nCategory Name values:")
print(df['Category Name'].value_counts())
print("\nDepartment Name values:")
print(df['Department Name'].value_counts())

Category Name unique values: 50
Department Name unique values: 11
Product Name unique values: 118

Category Name values:
Cleats                  24551
Men's Footwear          22246
Women's Apparel         21035
Indoor/Outdoor Games    19298
Fishing                 17325
Water Sports            15540
Camping & Hiking        13729
Cardio Equipment        12487
Shop By Sport           10984
Electronics              3156
Accessories              1780
Golf Balls               1475
Girls' Apparel           1201
Golf Gloves              1070
Trade-In                  974
Video Games               838
Children's Clothing       652
Women's Clothing          650
Baseball & Softball       632
Hockey                    614
Cameras                   592
Toys                      529
Golf Shoes                524
Pet Supplies              492
Garden                    484
Crafts                    484
DVDs                      483
Computers                 442
Golf Apparel              441
Hunting &

In [6]:
# Check order location cardinality
print("Order Country unique values:", df['Order Country'].nunique())
print("Order State unique values:", df['Order State'].nunique())
print("Order Region unique values:", df['Order Region'].nunique())
print("Market unique values:", df['Market'].nunique())
print("\nMarket values:")
print(df['Market'].value_counts())
print("\nOrder Region values:")
print(df['Order Region'].value_counts())

Order Country unique values: 164
Order State unique values: 1089
Order Region unique values: 23
Market unique values: 5

Market values:
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: Market, dtype: int64

Order Region values:
Central America    28341
Western Europe     27109
South America      14935
Oceania            10148
Northern Europe     9792
Southeast Asia      9539
Southern Europe     9431
Caribbean           8318
West of USA         7993
South Asia          7731
Eastern Asia        7280
East of USA         6915
West Asia           6009
US Center           5887
South of  USA       4045
Eastern Europe      3920
West Africa         3696
North Africa        3232
East Africa         1852
Central Africa      1677
Southern Africa     1157
Canada               959
Central Asia         553
Name: Order Region, dtype: int64


In [7]:
# Convert order date to datetime format
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

# Extract month
df['order_month'] = df['order date (DateOrders)'].dt.month

# Verifying the result
print("Order month sample:")
print(df['order_month'].value_counts().sort_index())

Order month sample:
1     17979
2     14529
3     15919
4     15435
5     15976
6     15139
7     15922
8     15912
9     15489
10    12955
11    12500
12    12764
Name: order_month, dtype: int64


In [8]:
# Final columns to keep
cols_to_keep = [
    'Days for shipment (scheduled)',
    'Late_delivery_risk',
    'Shipping Mode',
    'Type',
    'Order Item Discount Rate',
    'Order Item Product Price',
    'Order Item Quantity',
    'Product Price',
    'Product Status',
    'Customer Segment',
    'Category Name',
    'Department Name',
    'Market',
    'order_month'
]

# Create clean dataframe
df_clean = df[cols_to_keep].copy()

print("Clean dataframe shape:", df_clean.shape)
print("\nMissing values in clean data:")
print(df_clean.isnull().sum())
print("\nFirst 5 rows:")
df_clean.head()

Clean dataframe shape: (180519, 14)

Missing values in clean data:
Days for shipment (scheduled)    0
Late_delivery_risk               0
Shipping Mode                    0
Type                             0
Order Item Discount Rate         0
Order Item Product Price         0
Order Item Quantity              0
Product Price                    0
Product Status                   0
Customer Segment                 0
Category Name                    0
Department Name                  0
Market                           0
order_month                      0
dtype: int64

First 5 rows:


,Days for shipment (scheduled),Late_delivery_risk,Shipping Mode,Type,Order Item Discount Rate,Order Item Product Price,Order Item Quantity,Product Price,Product Status,Customer Segment,Category Name,Department Name,Market,order_month
0,4,0,Standard Class,DEBIT,0.04,327.75,1,327.75,0,Consumer,Sporting Goods,Fitness,Pacific Asia,1
1,4,1,Standard Class,TRANSFER,0.05,327.75,1,327.75,0,Consumer,Sporting Goods,Fitness,Pacific Asia,1
2,4,0,Standard Class,CASH,0.06,327.75,1,327.75,0,Consumer,Sporting Goods,Fitness,Pacific Asia,1
3,4,0,Standard Class,DEBIT,0.07,327.75,1,327.75,0,Home Office,Sporting Goods,Fitness,Pacific Asia,1
4,4,0,Standard Class,PAYMENT,0.09,327.75,1,327.75,0,Corporate,Sporting Goods,Fitness,Pacific Asia,1


In [10]:
# Encode categorical columns
df_encoded = pd.get_dummies(df_clean, 
                             columns=['Shipping Mode', 'Type', 
                                     'Customer Segment', 'Category Name',
                                     'Department Name', 'Market'],
                             drop_first=True)

print("Encoded shape:", df_encoded.shape)
print("\nFirst 10 columns:")
print(df_encoded.columns.tolist()[:10])

Encoded shape: (180519, 79)

First 10 columns:
['Days for shipment (scheduled)', 'Late_delivery_risk', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Quantity', 'Product Price', 'Product Status', 'order_month', 'Shipping Mode_Same Day', 'Shipping Mode_Second Class']


In [11]:
# Separate features (X) and target (y)
X = df_encoded.drop(columns=['Late_delivery_risk'])
y = df_encoded['Late_delivery_risk']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Features shape: (180519, 78)
Target shape: (180519,)

Target distribution:
1    98977
0    81542
Name: Late_delivery_risk, dtype: int64


In [13]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts())
print("Test target distribution:")
print(y_test.value_counts())

Training set size: (144415, 78)
Test set size: (36104, 78)

Training target distribution:
1    79180
0    65235
Name: Late_delivery_risk, dtype: int64
Test target distribution:
1    19797
0    16307
Name: Late_delivery_risk, dtype: int64


In [14]:
# Train Logistic Regression
print("Training Logistic Regression")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
print("Logistic Regression Done!")

# Train Random Forest
print("\nTraining Random Forest")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print("Random Forest Done!")

Training Logistic Regression
Logistic Regression Done!

Training Random Forest
Random Forest Done!


In [15]:
# Logistic Regression predictions
lr_predictions = lr_model.predict(X_test)

# Random Forest predictions
rf_predictions = rf_model.predict(X_test)

# Logistic Regression evaluation
print("=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, lr_predictions):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, lr_predictions))

# Random Forest evaluation
print("=" * 50)
print("RANDOM FOREST RESULTS")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, rf_predictions):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, rf_predictions))

LOGISTIC REGRESSION RESULTS
Accuracy: 69.22%

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.88      0.72     16307
           1       0.84      0.54      0.66     19797

    accuracy                           0.69     36104
   macro avg       0.73      0.71      0.69     36104
weighted avg       0.74      0.69      0.69     36104

RANDOM FOREST RESULTS
Accuracy: 64.40%

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.63      0.62     16307
           1       0.68      0.65      0.67     19797

    accuracy                           0.64     36104
   macro avg       0.64      0.64      0.64     36104
weighted avg       0.65      0.64      0.64     36104



In [16]:
# Check feature importance from Random Forest
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Top 15 most important features:")
print(feature_importance.head(15))

Top 15 most important features:
Order Item Discount Rate         0.332981
order_month                      0.162479
Days for shipment (scheduled)    0.104389
Shipping Mode_Standard Class     0.103204
Order Item Quantity              0.078249
Customer Segment_Corporate       0.024384
Customer Segment_Home Office     0.021283
Shipping Mode_Same Day           0.018703
Shipping Mode_Second Class       0.018563
Product Price                    0.014874
Order Item Product Price         0.014860
Market_Pacific Asia              0.011228
Type_DEBIT                       0.011202
Market_Europe                    0.010363
Type_TRANSFER                    0.010238
dtype: float64


In [17]:
# Tuned Random Forest
rf_model_tuned = RandomForestClassifier(
    n_estimators=100,
    max_features=0.3,    # each tree only sees 30% of features
    max_depth=15,         # limits tree depth
    random_state=42
)

rf_model_tuned.fit(X_train, y_train)
rf_tuned_predictions = rf_model_tuned.predict(X_test)

print("Tuned Random Forest Accuracy:")
print(f"{accuracy_score(y_test, rf_tuned_predictions):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, rf_tuned_predictions))

Tuned Random Forest Accuracy:
69.11%

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.86      0.71     16307
           1       0.82      0.55      0.66     19797

    accuracy                           0.69     36104
   macro avg       0.72      0.71      0.69     36104
weighted avg       0.73      0.69      0.69     36104



In [18]:
from sklearn.preprocessing import StandardScaler

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Retrain Logistic Regression on scaled data
lr_scaled = LogisticRegression(max_iter=1000)
lr_scaled.fit(X_train_scaled, y_train)
lr_scaled_predictions = lr_scaled.predict(X_test_scaled)

print("=" * 50)
print("LOGISTIC REGRESSION WITH SCALING")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, lr_scaled_predictions):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, lr_scaled_predictions))

LOGISTIC REGRESSION WITH SCALING
Accuracy: 69.18%

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.88      0.72     16307
           1       0.84      0.54      0.66     19797

    accuracy                           0.69     36104
   macro avg       0.73      0.71      0.69     36104
weighted avg       0.74      0.69      0.69     36104



In [20]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

print("=" * 50)
print("XGBOOST RESULTS")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, xgb_predictions):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, xgb_predictions))

XGBOOST RESULTS
Accuracy: 68.90%

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.85      0.71     16307
           1       0.82      0.56      0.66     19797

    accuracy                           0.69     36104
   macro avg       0.71      0.70      0.69     36104
weighted avg       0.72      0.69      0.68     36104



## Version 2 — Rebuilt and Improved
### More features · Smarter encoding · Better models

In [21]:
# Reload fresh copy of original data
df2 = df.copy()

# Parse order date and extract temporal features 
df2['order_date'] = pd.to_datetime(df2['order date (DateOrders)'], errors='coerce')
df2['order_month'] = df2['order_date'].dt.month
df2['order_dayofweek'] = df2['order_date'].dt.dayofweek  # Mon=0, Sun=6
df2['order_hour'] = df2['order_date'].dt.hour
df2['order_quarter'] = df2['order_date'].dt.quarter

print("Temporal features extracted")
print(df2[['order_month', 'order_dayofweek', 'order_hour', 'order_quarter']].head())

Temporal features extracted
   order_month  order_dayofweek  order_hour  order_quarter
0            1                2          22              1
1            1                5          12              1
2            1                5          12              1
3            1                5          11              1
4            1                5          11              1


In [22]:
# Columns to drop
cols_to_drop_v2 = [
    # Leakage columns
    'Days for shipping (real)',
    'Delivery Status',
    'Benefit per order',
    'Sales per customer',

    # Personal info
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',

    # High cardinality / not useful
    'Product Description',
    'Product Image',
    'Product Name',
    'Order Zipcode',
    'Customer Zipcode',

    # ID columns
    'Order Id',
    'Order Item Id',
    'Order Customer Id',
    'Order Item Cardprod Id',
    'Product Card Id',
    'Product Category Id',
    'Customer Id',
    'Department Id',
    'Category Id',

    # Date strings — already extracted what we need
    'order date (DateOrders)',
    'shipping date (DateOrders)',
    'order_date',

    # Redundant
    'order_quarter',
]

df2 = df2.drop(columns=cols_to_drop_v2)

print("Cleaned shape:", df2.shape)
print("\nRemaining columns:")
print(df2.columns.tolist())

Cleaned shape: (180519, 31)

Remaining columns:
['Type', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Price', 'Product Status', 'Shipping Mode', 'order_month', 'order_dayofweek', 'order_hour']


In [23]:
# Drop additional columns
additional_drops = [
    'Order Status',        # data leakage
    'Order Item Discount', # redundant with Discount Rate
    'Latitude',            # too granular, redundant
    'Longitude',           # too granular, redundant
]

df2 = df2.drop(columns=additional_drops)
print("Updated shape:", df2.shape)
print("\nFinal columns:")
print(df2.columns.tolist())

Updated shape: (180519, 27)

Final columns:
['Type', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Product Price', 'Product Status', 'Shipping Mode', 'order_month', 'order_dayofweek', 'order_hour']


In [24]:
# correlation between financial columns
financial_cols = ['Sales', 'Order Item Total', 'Order Item Product Price', 
                  'Product Price', 'Order Profit Per Order', 'Order Item Profit Ratio']

print("Correlation matrix:")
print(df2[financial_cols].corr().round(2))

Correlation matrix:
                          Sales  Order Item Total  Order Item Product Price  \
Sales                      1.00              0.99                      0.79   
Order Item Total           0.99              1.00                      0.78   
Order Item Product Price   0.79              0.78                      1.00   
Product Price              0.79              0.78                      1.00   
Order Profit Per Order     0.13              0.13                      0.10   
Order Item Profit Ratio   -0.00             -0.00                     -0.00   

                          Product Price  Order Profit Per Order  \
Sales                              0.79                    0.13   
Order Item Total                   0.78                    0.13   
Order Item Product Price           1.00                    0.10   
Product Price                      1.00                    0.10   
Order Profit Per Order             0.10                    1.00   
Order Item Profit Ratio 

In [25]:
# Drop highly correlated redundant columns
cols_to_drop_corr = [
    'Order Item Total',          # 0.99 correlated with Sales
    'Order Item Product Price',  # 1.00 correlated with Product Price
    'Order Profit Per Order',    # 0.82 correlated with Profit Ratio
]

df2 = df2.drop(columns=cols_to_drop_corr)

print("Updated shape:", df2.shape)
print("\nFinal columns:")
print(df2.columns.tolist())

Updated shape: (180519, 24)

Final columns:
['Type', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Item Discount Rate', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Region', 'Order State', 'Product Price', 'Product Status', 'Shipping Mode', 'order_month', 'order_dayofweek', 'order_hour']


In [26]:
from sklearn.preprocessing import LabelEncoder

# Separate target
y2 = df2['Late_delivery_risk']
X2 = df2.drop(columns=['Late_delivery_risk'])

# Identify categorical columns
cat_cols = X2.select_dtypes(include=['object']).columns.tolist()
num_cols = X2.select_dtypes(include=['number']).columns.tolist()

print("Categorical columns:", len(cat_cols))
print(cat_cols)
print("\nNumeric columns:", len(num_cols))
print(num_cols)

# Label encode categorical columns
le = LabelEncoder()
for col in cat_cols:
    X2[col] = X2[col].astype(str).fillna('MISSING')
    X2[col] = le.fit_transform(X2[col])

print("\nEncoding complete")
print("Shape:", X2.shape)

Categorical columns: 13
['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Shipping Mode']

Numeric columns: 10
['Days for shipment (scheduled)', 'Order Item Discount Rate', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Product Price', 'Product Status', 'order_month', 'order_dayofweek', 'order_hour']

Encoding complete
Shape: (180519, 23)


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# Split data
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

# Scaled version for Logistic Regression
scaler2 = RobustScaler()
X2_train_sc = scaler2.fit_transform(X2_train)
X2_test_sc  = scaler2.transform(X2_test)

print(f"Train: {X2_train.shape} | Test: {X2_test.shape}")
print(f"\nClass balance (train):")
print(y2_train.value_counts(normalize=True).round(3))

Train: (144415, 23) | Test: (36104, 23)

Class balance (train):
1    0.548
0    0.452
Name: Late_delivery_risk, dtype: float64


In [30]:
#install lightgbm
!pip install lightgbm

     |████████████████████████████████| 2.0 MB 3.9 MB/s eta 0:00:01


In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import xgboost as xgb
import lightgbm as lgb

# Results storage
results = []

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, average='weighted')
    auc = roc_auc_score(y_te, y_prob)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'F1': f1,
        'ROC-AUC': auc
    })
    print(f"{name:<30} Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")
    return model

print("=" * 65)
print(f"{'Model':<30} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print("=" * 65)

# Logistic Regression — uses scaled data
lr_v2  = evaluate_model(
    "Logistic Regression",
    LogisticRegression(max_iter=1000, random_state=42),
    X2_train_sc, X2_test_sc, y2_train, y2_test
)

rf_v2  = evaluate_model(
    "Random Forest",
    RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    X2_train, X2_test, y2_train, y2_test
)

xgb_v2 = evaluate_model(
    "XGBoost",
    xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, n_jobs=-1
    ),
    X2_train, X2_test, y2_train, y2_test
)

lgb_v2 = evaluate_model(
    "LightGBM",
    lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    ),
    X2_train, X2_test, y2_train, y2_test
)

Model                               Acc       F1      AUC
Logistic Regression            Acc=0.6901  F1=0.6885  AUC=0.7268
Random Forest                  Acc=0.7219  F1=0.7183  AUC=0.8298
XGBoost                        Acc=0.7292  F1=0.7265  AUC=0.8229
LightGBM                       Acc=0.7290  F1=0.7263  AUC=0.8184


In [32]:
# Full report for best model — XGBoost
print("=" * 50)
print("BEST MODEL — XGBoost Full Report")
print("=" * 50)
xgb_predictions_v2 = xgb_v2.predict(X2_test)
print(classification_report(y2_test, xgb_predictions_v2))

# Feature importance
print("\nTop 15 Most Important Features:")
xgb_importance = pd.Series(
    xgb_v2.feature_importances_,
    index=X2_train.columns
).sort_values(ascending=False)
print(xgb_importance.head(15))

BEST MODEL — XGBoost Full Report
              precision    recall  f1-score   support

           0       0.65      0.88      0.75     16308
           1       0.86      0.61      0.71     19796

    accuracy                           0.73     36104
   macro avg       0.75      0.74      0.73     36104
weighted avg       0.76      0.73      0.73     36104


Top 15 Most Important Features:
Days for shipment (scheduled)    0.442420
Shipping Mode                    0.392926
Type                             0.035068
order_hour                       0.031618
Order State                      0.006815
Order City                       0.006749
Order Country                    0.006691
Customer State                   0.006649
Customer City                    0.006587
Order Region                     0.006452
order_month                      0.006436
order_dayofweek                  0.006285
Customer Segment                 0.005978
Market                           0.005951
Customer Country   

In [34]:
import pickle

# Save XGBoost model
with open('xgb_supply_chain_model.pkl', 'wb') as f:
    pickle.dump(xgb_v2, f)

print("Model saved successfully!")

Model saved successfully!


In [36]:
# Create a dataframe with predictions for Tableau
tableau_df = X2_test.copy()
tableau_df['Actual'] = y2_test.values
tableau_df['Predicted'] = xgb_v2.predict(X2_test)
tableau_df['Probability_Late'] = xgb_v2.predict_proba(X2_test)[:, 1]

# Add readable labels
tableau_df['Actual_Label'] = tableau_df['Actual'].map({0: 'On Time', 1: 'Late'})
tableau_df['Predicted_Label'] = tableau_df['Predicted'].map({0: 'On Time', 1: 'Late'})
tableau_df['Correct_Prediction'] = (tableau_df['Actual'] == tableau_df['Predicted']).astype(int)

# Add original text labels back for Tableau
tableau_df['Shipping_Mode_Label'] = df['Shipping Mode'].iloc[y2_test.index].values
tableau_df['Market_Label'] = df['Market'].iloc[y2_test.index].values
tableau_df['Customer_Segment_Label'] = df['Customer Segment'].iloc[y2_test.index].values
tableau_df['Department_Label'] = df['Department Name'].iloc[y2_test.index].values

# Save to CSV
tableau_df.to_csv(os.path.expanduser('~/Desktop/supply_chain_predictions.csv'), index=False)

print("Tableau export complete! ✅")
print("Shape:", tableau_df.shape)
print("\nSample:")
print(tableau_df[['Actual_Label', 'Predicted_Label', 'Probability_Late', 
                   'Correct_Prediction', 'Shipping_Mode_Label', 'Market_Label']].head())

Tableau export complete! ✅
Shape: (36104, 33)

Sample:
       Actual_Label Predicted_Label  Probability_Late  Correct_Prediction  \
75083       On Time         On Time          0.383368                   1   
88208       On Time         On Time          0.256870                   1   
161090      On Time         On Time          0.324668                   1   
151346      On Time         On Time          0.015496                   1   
104796         Late            Late          0.977128                   1   

       Shipping_Mode_Label Market_Label  
75083       Standard Class         USCA  
88208       Standard Class        LATAM  
161090      Standard Class       Europe  
151346            Same Day         USCA  
104796            Same Day       Europe  
